# 02 · Medicine details and drug features (MedSafe-AI · Component 3)

**What this notebook does**
1. Reads your 14 openFDA label files and collects, for every drug: **drug class**, **main use** and the **side-effects text** from the label. (This is the "what is this medicine" part of your component.)
2. Gets each drug's chemical structure (**SMILES**) from PubChem. *This step needs internet.*
3. Turns every structure into numbers (**Morgan fingerprint**, needs RDKit).
4. Builds the final table of drug pairs that the models will use.

**Time:** label files about 5–15 minutes, PubChem about 10 minutes. Both steps **remember their progress**: if you stop, run again and they continue.

In [ ]:
import sys, importlib.util, subprocess
from pathlib import Path

HERE = Path.cwd()                                   # the folder of this notebook (component3)
if not (HERE / "c3_utils.py").exists():
    if Path("/content").exists():                   # Google Colab: get the code from GitHub
        REPO_DIR = Path("/content/repo")
        if not (REPO_DIR / "component3" / "c3_utils.py").exists():
            subprocess.run(["rm", "-rf", str(REPO_DIR)])
            subprocess.check_call(["git", "clone", "-q", "--depth", "1", "https://github.com/sksivakajan/mac-doc.git", str(REPO_DIR)])
        HERE = REPO_DIR / "component3"
        import os; os.chdir(HERE)
        print("Colab: code taken from GitHub ->", HERE)
    else:                                           # Jupyter was started from another folder
        HERE = Path(r"D:\med\data set\component3")
sys.path.insert(0, str(HERE))

def need(*pairs):
    """Install the packages that are missing. pairs = (import name, pip name)."""
    missing = [pip for mod, pip in pairs if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", missing)
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
        except Exception as e:
            print("!! Could not install", missing, "->", e)

need(("pandas", "pandas"), ("numpy", "numpy"), ("rdkit", "rdkit"))

import time
import numpy as np, pandas as pd
import c3_utils as c3

try:
    DATA_DIR = c3.find_data_dir()
except FileNotFoundError:
    DATA_DIR = None                       # e.g. in Google Colab: the saved drug_details_raw.csv is used instead
WORK = HERE / "work"
LABEL_FILES = sorted(DATA_DIR.glob("drug-label-*.json")) if DATA_DIR else []
print("Label files found:", len(LABEL_FILES), "(14 on your PC; 0 is fine in Colab)")
print("RDKit available  :", c3.have_rdkit())
print("Python in use     :", sys.executable)

## Step 1 · Settings

In [ ]:
N_BITS = 1024          # length of the fingerprint of one drug
RADIUS = 2             # Morgan radius (2 is the normal choice)
TOP_CLASSES = 100      # how many drug classes to use as features
MAX_LABELS = None      # set to e.g. 20000 for a quick test run; None = read everything

## Step 2 · Drug class, use and label side effects (openFDA labels)
Each label file is about 600 MB, so they are read as a **stream** (not loaded all at once).
Only labels with **one** active ingredient are used, so the class belongs clearly to that drug.

In [ ]:
drugs = pd.read_csv(WORK / "drugs_raw.csv")

# names we look for: exact name first, then the name without salt words (citrate, sodium ...)
wanted = {}
for did, nm in zip(drugs["drug_id"], drugs["drug_name"]):
    wanted.setdefault(c3.norm_name(nm), did)
for did, nm in zip(drugs["drug_id"], drugs["drug_name"]):
    for v in c3.name_variants(nm):
        wanted.setdefault(v, did)

DETAILS_FILE = WORK / "drug_details_raw.csv"
if DETAILS_FILE.exists() and MAX_LABELS is None:
    details = pd.read_csv(DETAILS_FILE, keep_default_na=False)
    print("Loaded saved details:", DETAILS_FILE.name)
else:
    if not LABEL_FILES:
        raise FileNotFoundError("No openFDA label files and no saved drug_details_raw.csv in the work folder.")
    t0 = time.time()
    details = c3.scan_labels(LABEL_FILES, wanted, max_labels=MAX_LABELS)
    if MAX_LABELS is None:
        details.to_csv(DETAILS_FILE, index=False)
    print(f"Done in {(time.time()-t0)/60:.1f} minutes")

drugs = drugs.merge(details, on="drug_id", how="left").fillna({"drug_class": "", "all_classes": "", "main_use": "",
                                                               "label_adverse_reactions": "", "n_labels": 0})
print(f"drugs with a label match : {(drugs['n_labels'] > 0).sum():,} of {len(drugs):,}")
print(f"drugs with a drug class  : {(drugs['drug_class'] != '').sum():,}")
print(f"drugs with a main use    : {(drugs['main_use'] != '').sum():,}")
display(drugs[drugs["drug_class"] != ""][["drug_name", "drug_class", "main_use"]].head(8))

## Step 3 · Chemical structure (SMILES) from PubChem
PubChem allows about 5 requests per second, so ~1,800 drugs take around 10 minutes. If PubChem is busy, the cell waits and tries again. Temporary failures are **never** saved as "not found"; if some remain, just run the cell again.
Only drugs PubChem really does not know (for example mixtures such as *Coal Tar*) are listed and **left out** of the models (we can not make a fingerprint for them).

In [ ]:
FP_FILE = WORK / "drug_fp_colab.npz"      # made in Google Colab: lets a PC without RDKit go on
if FP_FILE.exists():
    z = np.load(FP_FILE, allow_pickle=False)
    assert int(z["n_bits"]) == N_BITS and int(z["radius"]) == RADIUS, "the Colab fingerprint file has other settings (N_BITS / RADIUS)"
    drugs["smiles"] = drugs["drug_id"].map(dict(zip(z["ids"].tolist(), z["smiles"].tolist()))).fillna("")
    drugs["smiles_status"] = drugs["drug_id"].map(dict(zip(z["ids"].tolist(), z["status"].tolist()))).fillna("unknown")
    print("Using the structures and fingerprints made in Colab:", FP_FILE.name)
else:
    drugs = c3.get_smiles(drugs, WORK / "smiles_cache.csv")
print(drugs["smiles_status"].value_counts().to_dict())
have = drugs["smiles"] != ""
print(f"SMILES found: {have.sum():,} of {len(drugs):,} ({100*have.mean():.1f}%)")
drugs[~have][["drug_id", "drug_name", "smiles_status"]].to_csv(WORK / "drugs_without_smiles.csv", index=False)
print("Examples PubChem does not know:", list(drugs[drugs["smiles_status"] == "not_found"]["drug_name"].head(12)))

unknown = (drugs["smiles_status"] == "unknown").sum()
if unknown:
    if unknown <= 0.10 * len(drugs):
        print(f"NOTE: {unknown} drugs were not checked (PubChem was busy). They get no fingerprint now; "
              "the models still work. Later, run Step 2b again to ask only for those, then re-run from here.")
    else:
        raise RuntimeError(f"{unknown} drugs could not be checked (PubChem busy or internet problem). "
                           "Run the Step 2b cell again: it only asks PubChem for the missing ones.")

## Step 4 · Fingerprints
A Morgan fingerprint is a 0/1 list that describes which small chemical pieces a drug contains.
Drugs with similar structure get similar fingerprints, so the model can guess for drugs it has seen little.

In [ ]:
if FP_FILE.exists():
    idmap = {i: k for k, i in enumerate(z["ids"].tolist())}
    drugs["has_fp"] = [(i in idmap) and bool(z["valid"][idmap[i]]) for i in drugs["drug_id"]]
    print(f"valid fingerprints: {int(drugs['has_fp'].sum()):,} of {len(drugs):,}")
    drugs_final = drugs[drugs["has_fp"]].reset_index(drop=True)
    FP = z["fp"][[idmap[i] for i in drugs_final["drug_id"]]].astype(np.uint8)
    print("fingerprint matrix:", FP.shape, "  average bits set per drug:", round(float(FP.sum(1).mean()), 1))
else:
    if not c3.have_rdkit():
        err = c3.rdkit_error()
        print("Python in use:", sys.version.split()[0], "-", sys.executable)
        print("RDKit problem:", err)
        if "Application Control" in err or "blocked" in err.lower():
            raise RuntimeError(
                "Windows is BLOCKING RDKit on this computer (Application Control / Smart App Control). "
                "Do not try to work around it. Use the notebook  02b_fingerprints_colab.ipynb  in Google Colab "
                "(instructions are inside it), unzip its result into the work folder, then go on with 03.")
        print("Trying to install RDKit now ...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "rdkit"])
        except Exception as e:
            print("pip install failed:", e)
    if not c3.have_rdkit():
        raise RuntimeError(
            "RDKit is still not available: " + c3.rdkit_error() + "\n"
            "Try in a new cell:   %pip install rdkit   then run this cell again.\n"
            "If it still fails use  02b_fingerprints_colab.ipynb  in Google Colab.")

    fps, ok = [], []
    for smi in drugs["smiles"]:
        fp = c3.morgan_fp(smi, RADIUS, N_BITS) if smi else None
        ok.append(fp is not None)
        fps.append(fp)
    drugs["has_fp"] = ok
    print(f"valid fingerprints: {sum(ok):,} of {len(drugs):,}")

    drugs_final = drugs[drugs["has_fp"]].reset_index(drop=True)
    FP = np.stack([f for f in fps if f is not None]).astype(np.uint8)
    print("fingerprint matrix:", FP.shape, "  average bits set per drug:", round(float(FP.sum(1).mean()), 1))

## Step 5 · Drug classes as numbers

In [ ]:
class_lists = [[c for c in str(s).split("; ") if c] for s in drugs_final["all_classes"]]
CM, class_names = c3.class_matrix(class_lists, TOP_CLASSES)
print("class matrix:", CM.shape, "  drugs with at least one class:", int((CM.sum(1) > 0).sum()))
print("most common classes:", class_names[:8])

## Step 6 · Final pair table for the models
We keep only pairs where **both** drugs have a fingerprint. Pairs are stored with the **row number** of each drug (`idx_a`, `idx_b`).

In [ ]:
pairs = pd.read_csv(WORK / "pairs_raw.csv")
Y = np.load(WORK / "Y_raw.npy")
effects = pd.read_csv(WORK / "effects.csv")

row_of = {int(d): i for i, d in enumerate(drugs_final["drug_id"])}
ia = pairs["drug_a_id"].map(row_of)
ib = pairs["drug_b_id"].map(row_of)
keep = ia.notna() & ib.notna()
print(f"pairs before: {len(pairs):,}   pairs with both drugs usable: {keep.sum():,} ({100*keep.mean():.1f}%)")

pairs_f = pairs[keep].copy()
pairs_f["idx_a"] = ia[keep].astype(int).values
pairs_f["idx_b"] = ib[keep].astype(int).values
Y_f = Y[keep.values]

# an effect that lost every positive can not be learned -> report it
zero = effects[Y_f.sum(0) == 0]["effect_name"].tolist()
print("side effects with no positive pair left:", zero if zero else "none")

pairs_f.reset_index(drop=True).to_csv(WORK / "modeling_pairs.csv", index=False)
np.save(WORK / "modeling_Y.npy", Y_f)
np.savez_compressed(WORK / "drug_features.npz", FP=FP, CM=CM, class_names=np.array(class_names))
drugs_final.drop(columns=["has_fp"]).to_csv(WORK / "drugs_final.csv", index=False)
effects.to_csv(WORK / "modeling_effects.csv", index=False)
print(f"saved: {len(pairs_f):,} pairs, {len(drugs_final):,} drugs, {Y_f.shape[1]} side effects")

## Done ✔
**Next:** open `03_split_and_check.ipynb` to make the 70 / 20 / 10 split and check it.